## Step 1:
- Load the pandas library so I can work with tables of data

In [1]:
import pandas as pd

## Step 2:
This is a reusable function that cleans one commodity's data:
- combines the 3 yearly files into one
- keeps only the useful columns
- removes organic listings (keeping only conventional, to avoid price differences from that)
- picks one price per row (uses the "typical" price when available, otherwise falls back to the average of the low/high range)
- removes any rows that still have no price at all

In [10]:
def clean_commodity(commodity_name, filenames):
    dfs = [pd.read_csv(f'../data/raw/{f}') for f in filenames]
    df = pd.concat(dfs, ignore_index=True)
    df['report_date'] = pd.to_datetime(df['report_date'])
    
    columns_to_keep = ['report_date', 'location', 'group', 'commodity', 'variety',
                        'package', 'origin', 'item_size', 'organic',
                        'low_price', 'high_price', 'mostly_low_price', 'mostly_high_price']
    df = df[columns_to_keep].copy()
    df = df[df['organic'] == 'N'].copy()
    
    df['rep_low'] = df['mostly_low_price'].fillna(df['low_price'])
    df['rep_high'] = df['mostly_high_price'].fillna(df['high_price'])
    df['rep_price'] = (df['rep_low'] + df['rep_high']) / 2
    df = df.dropna(subset=['rep_price'])
    
    df['commodity_name'] = commodity_name
    return df

## Step 3:
- Created a list telling the code which files belong to which commodity
- This will be used to pass into the cleaning function

In [20]:
commodity_files = {
    "Avocados": ["avocados_23.csv", "avocados_24.csv", "avocados_25.csv"],
    "Tomatoes": ["tomatoes_23.csv", "tomatoes_24.csv", "tomatoes_25.csv"],
    "Romaine Lettuce": ["lettuceRomaine_23.csv", "lettuceRomaine_24.csv", "lettuceRomaine_25.csv"],
    "Strawberries": ["strawberries_23.csv", "strawberries_24.csv", "strawberries_25.csv"],
    "Bananas": ["bananas_23.csv", "bananas_24.csv", "bananas_25.csv"],
}

## Step 4:
- Runs the cleaning steps (from Cell 2) on all 5 commodities, one at a time
- Then combines all 5 cleaned results into one big table

In [13]:
all_cleaned = []
for name, files in commodity_files.items():
    cleaned = clean_commodity(name, files)
    print(f"{name}: {cleaned.shape[0]} rows")
    all_cleaned.append(cleaned)

combined_all = pd.concat(all_cleaned, ignore_index=True)
print("Total combined:", combined_all.shape)

Avocados: 12076 rows
Tomatoes: 9178 rows
Romaine Lettuce: 3225 rows
Strawberries: 5584 rows
Bananas: 19124 rows
Total combined: (49187, 17)


## Step 5:
- Groups the data down to one row per commodity, per date, per origin
- Calculates the average, lowest, and highest price for that day, plus how many listings made up that average
- Also removes rows where origin was just labeled "Imports" with no specific country given

In [14]:
daily_by_origin = (
    combined_all.groupby(['commodity_name', 'report_date', 'origin'])
    .agg(
        avg_price=('rep_price', 'mean'),
        min_price=('rep_price', 'min'),
        max_price=('rep_price', 'max'),
        num_listings=('rep_price', 'count')
    )
    .reset_index()
)

daily_by_origin = daily_by_origin[daily_by_origin['origin'] != 'Imports']
print(daily_by_origin.shape)

(8082, 7)


## Step 6:
- Saved the cleaned daily price table (built in Step 5) to a file
- This file will be loaded into SQL later

In [19]:
import os
os.makedirs('../data/cleaned', exist_ok=True)

daily_by_origin.to_csv('../data/cleaned/daily_prices.csv', index=False)
print("Saved:", daily_by_origin.shape)

Saved: (8082, 7)


## Step 7: 
- Creates a simple list of the 5 commodities
- This will be used as a lookup table in SQL

In [17]:
commodities = pd.DataFrame({
    "commodity_name": ["Avocados", "Tomatoes", "Romaine Lettuce", "Strawberries", "Bananas"]
})
commodities.to_csv('../data/cleaned/commodities.csv', index=False)
commodities

,commodity_name
0,Avocados
1,Tomatoes
2,Romaine Lettuce
3,Strawberries
4,Bananas


## Step 8:
- Creates a list of each origin (country/state)
- This will also be used as a lookup table in SQL

In [18]:
origins = pd.DataFrame({
    "origin": ["Mexico", "California", "Guatemala", "Ecuador", "Colombia", 
               "Costa Rica", "Peru", "Florida", "Arizona-California", 
               "Canada", "Dominican Republic", "New Zealand"]
})
origins.to_csv('../data/cleaned/origins.csv', index=False)
origins

,origin
0,Mexico
1,California
2,Guatemala
3,Ecuador
4,Colombia
5,Costa Rica
6,Peru
7,Florida
8,Arizona-California
9,Canada
